# ENEC 2026 · Day 2 · GenAI Basics and RAG From Scratch

**Time:** about 75 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Understand tokens, context and hallucination, then build a working retrieval-augmented assistant over the A-001 documents with no framework: chunk, embed, retrieve, answer with citations.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai tiktoken

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enec2026oct.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enec2026oct"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enec2026oct" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enec2026oct.zip in the Files panel (left sidebar), "
            "run  !unzip -q enec2026oct.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

## 1. Tokens and context

An LLM does not read words. It reads **tokens** (pieces of words). The **context window** is how many tokens it can consider at once, including your prompt and its answer. Cost and speed depend on tokens.

In [ ]:
import tiktoken, pandas as pd, numpy as np, json, re
enc = tiktoken.get_encoding("o200k_base")

print(enc.encode("Pump A-001 vibration"))
print([enc.decode([t]) for t in enc.encode("Pump A-001 vibration is rising")])

docs = enec.load_documents()
pd.DataFrame([{"doc_id": d["doc_id"], "status": d["status"], "version": d["version"],
               "characters": len(d["text"]), "tokens": len(enc.encode(d["text"]))} for d in docs])

**Question:** The nine documents total only a few thousand tokens. Why not paste them all into every prompt? (Think: 9 documents today, 9,000 tomorrow. Cost, latency, and the model losing the needle in the haystack.) **RAG** retrieves only the few relevant pieces.

## 2. Why we need grounding: the hallucination test

In [ ]:
q = "Is the A-001 inspection procedure version 3.0 approved and in force?"
print(enec.ask(client, q))

The model has no access to our documents, so any confident answer is a guess. Now we build the retrieval pipeline step by step.

## 3. The documents and their governance metadata

Every document has a **status**. This matters later: a newer document is not necessarily the current authority.

In [ ]:
pd.DataFrame([{k: d[k] for k in ("doc_id", "title", "status", "version", "date")} for d in docs])

## 4. Chunking

We split each document into pieces of about 900 characters (with a little overlap) so each piece holds one idea. Too big and the search is vague. Too small and the context is lost.

In [ ]:
def chunk_text(text, size=900, overlap=150):
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 2 <= size:
            cur = (cur + "\n\n" + p).strip()
        else:
            if cur: chunks.append(cur)
            tail = cur[-overlap:] if cur else ""
            cur = (tail + "\n\n" + p).strip()
    if cur: chunks.append(cur)
    return chunks

chunks = []
for d in docs:
    for i, c in enumerate(chunk_text(d["text"])):
        chunks.append({"doc_id": d["doc_id"], "title": d["title"], "status": d["status"],
                       "version": d["version"], "chunk_no": i, "text": c})
print(len(chunks), "chunks")
print(chunks[3]["doc_id"], "chunk", chunks[3]["chunk_no"], "\n")
print(chunks[3]["text"][:500])

## 5. Embeddings

An **embedding** turns text into a list of numbers so that texts with similar meaning end up close together. We embed every chunk once.

In [ ]:
texts = [f"{c['title']} ({c['status']} v{c['version']})\n{c['text']}" for c in chunks]
vectors = enec.embed(client, texts)
vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9)
print("vectors:", vectors.shape, "(chunks x dimensions)")

## 6. Retrieval: cosine similarity

Embed the question the same way, then rank chunks by similarity. Because vectors are normalised, cosine similarity is just a dot product.

In [ ]:
def retrieve(question, k=4):
    qv = enec.embed(client, [question])[0]
    qv = qv / (np.linalg.norm(qv) + 1e-9)
    scores = vectors @ qv
    top = np.argsort(-scores)[:k]
    return [{**chunks[i], "score": float(scores[i])} for i in top]

for h in retrieve("Which inspection procedure is currently approved for A-001?"):
    print(f"{h['score']:.3f}  {h['doc_id']:<24} {h['status']:<11} {h['text'][:90]!r}")

## 7. Generate a grounded answer with citations

The prompt has three parts: **rules**, **evidence** (labelled with document ID, status, version) and the **question**.

In [ ]:
RULES = """You answer questions about synthetic asset A-001 using ONLY the EVIDENCE.
- Cite document IDs in square brackets, e.g. [A001-PROC-INS-001-V2].
- Only APPROVED documents are current authority. DRAFT is not authoritative. SUPERSEDED is historical only.
- If the evidence does not answer the question, say so. Do not guess.
- You support a human reviewer. You never authorise work."""

def answer(question, k=4):
    hits = retrieve(question, k)
    evidence = "\n\n---\n\n".join(
        f"[{h['doc_id']} | {h['status']} | v{h['version']}]\n{h['text']}" for h in hits)
    reply = enec.ask(client, f"EVIDENCE:\n{evidence}\n\nQUESTION: {question}", system=RULES)
    return reply, hits

reply, hits = answer("Is the A-001 inspection procedure version 3.0 approved and in force?")
print(reply)
print("\nSources:", sorted({h["doc_id"] for h in hits}))

Compare this with the ungrounded answer in section 2.

## 8. Ask more questions

In [ ]:
for q in ["What did the field observer report on 19 August, and was the source of the wet area confirmed?",
          "Does work order WO-2026-0817 authorise component replacement?",
          "What should be checked when vibration is rising?"]:
    r, h = answer(q)
    print("Q:", q); print(r); print("Sources:", [x["doc_id"] for x in h], "\n" + "-" * 70)

## 9. What do the retrieval settings change?

Try different chunk sizes and values of k. We only run retrieval here (cheap), and look at *which documents* come back for one question.

In [ ]:
question = "What does the approved procedure say about evidence to review before classifying the condition?"

def build(size, overlap=150):
    cs = []
    for d in docs:
        for i, c in enumerate(chunk_text(d["text"], size, overlap)):
            cs.append({"doc_id": d["doc_id"], "status": d["status"], "text": c})
    v = enec.embed(client, [c["text"] for c in cs]); v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
    return cs, v

rows = []
qv = enec.embed(client, [question])[0]; qv = qv / np.linalg.norm(qv)
for size in (400, 900, 1600):
    cs, v = build(size)
    top = np.argsort(-(v @ qv))[:4]
    rows.append({"chunk_size": size, "chunks": len(cs), "top4_docs": [cs[i]["doc_id"] for i in top]})
pd.DataFrame(rows)

## Takeaways

- RAG = **chunk, embed, retrieve, generate**. No framework is required to understand it.
- Retrieval quality decides answer quality. Always look at the retrieved chunks.
- Label evidence with **ID, status and version**, and tell the model how to treat each status.
- The model can still be wrong. Citations let a human check.

## Exercise

1. Ask a question the documents cannot answer (for example: "What is the replacement cost of the A-001 bearing?"). Does the assistant say it does not know?
2. Change `RULES` to remove the "do not guess" line. Re-ask. What changes?
3. Change `k` from 4 to 1 and to 8. When does the answer get worse?